# Modelos LLM e API Hugging Face

Explorando a API de baixo nível do Hugging Face Transformers — os modelos PyTorch que implementam a arquitetura dos transformers.

Este notebook pode ser executado em um ambiente de execução gratuito ou de baixo custo com GPU T4 no Colab ou localmente com suporte a GPU.


## Lembrete importante sobre o Google Colab

**Dica valiosa:**

No meio da execução do Colab, você pode receber um erro como este:

> Runtime error: CUDA is required but not available for bitsandbytes. Please consider installing [...]

Esta é uma mensagem de erro enganosa! Não tente alterar as versões dos pacotes...

Isso acontece quando o Colab altera o ambiente de execução. A solução é:

1. Menu **Ambiente de execução** (Runtime) >> **Desconectar e excluir ambiente de execução**
2. Recarregue a página do Colab e vá em **Editar** >> **Limpar todas as saídas**
3. Conecte-se a uma nova T4 usando o botão no canto superior direito
4. Selecione **Ver recursos** no menu superior direito para confirmar que você possui uma GPU
5. Execute novamente as células do notebook do topo para baixo, começando pelas instalações com `pip`


In [20]:
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6

In [21]:
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
import torch
import gc

# Autenticação no Hugging Face

1. Se você ainda não fez isso, crie uma conta gratuita no HuggingFace em https://huggingface.co e navegue até **Settings** >> **Access Tokens**, criando um novo token de API com permissão de escrita (aba **WRITE**).

2. Pressione o ícone de chave (Secrets) no painel esquerdo do Colab e adicione um novo segredo:
`HF_TOKEN = seu_token`

3. Execute a célula abaixo para realizar o login.

In [22]:
hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

### Acesso aos Modelos Llama

Para utilizar os modelos Llama da Meta, você precisa aceitar os termos de licença no Hugging Face:

- Llama 3.1 8B Instruct: https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct
- Llama 3.2 1B Instruct (mais leve e rápido): https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct

Selecione os links acima para solicitar acesso caso ainda não tenha feito. Escolha a versão do Llama que deseja utilizar no código abaixo desmarcando o comentário.

In [23]:
# Modelos do tipo instruct e 1 modelo de raciocínio (reasoning)

# Llama 3.1 é maior (requer permissão de acesso no HF)
# LLAMA = "meta-llama/Meta-Llama-3.1-8B-Instruct"

# Llama 3.2 é menor e mais rápido para testes
LLAMA = "meta-llama/Llama-3.2-1B-Instruct"

PHI = "microsoft/Phi-4-mini-instruct"
GEMMA = "google/gemma-3-270m-it"
QWEN = "Qwen/Qwen3-4B-Instruct-2507"
DEEPSEEK = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

In [24]:
mensagens = [
    {"role": "user", "content": "Conte uma piada engraçada para uma sala de Cientistas de Dados"}
  ]

# Acessando o Llama 3.1 da Meta

Para usar o Llama 3.1, a Meta exige que você aceite os termos de serviço.

Visite a página do modelo no Hugging Face:
https://huggingface.co/meta-llama/Meta-Llama-3.1-8B

No topo da página estão as instruções para aceitar os termos. Se possível, use o mesmo e-mail da sua conta do Hugging Face.

A aprovação geralmente leva poucos minutos. Assim que for aprovado para qualquer modelo 3.1, a permissão é válida para toda a família de modelos.

In [25]:
# Configuração de Quantização - permite carregar o modelo na memória consumindo menos VRAM

config_quantizacao = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

Se a próxima célula retornar um erro de permissão (403), verifique:
1. Você está autenticado no HuggingFace? Execute `login()` para testar sua chave;
2. Seu token de API foi configurado com permissões de leitura e escrita?
3. Ao visitar a página do Llama 3.1 em https://huggingface.co/meta-llama/Meta-Llama-3.1-8B, mostra que você tem acesso ao modelo no topo da página?

In [26]:
# Tokenizador

tokenizador = AutoTokenizer.from_pretrained(LLAMA)
tokenizador.pad_token = tokenizador.eos_token
entradas = tokenizador.apply_chat_template(mensagens, return_tensors="pt").to("cuda")

In [27]:
entradas

tensor([[128000, 128006,   9125, 128007,    271,  38766,   1303,  33025,   2696,
             25,   6790,    220,   2366,     18,    198,  15724,   2696,     25,
            220,   1721,   5033,    220,   2366,     21,    271, 128009, 128006,
            882, 128007,    271,    825,     68,  10832,   9115,   2649,    665,
          33050,   3209,   2649,   3429,  10832,  59033,    409,    356,   1188,
          27771,    409,    423,   5670, 128009]], device='cuda:0')

In [28]:
# O Modelo

modelo = AutoModelForCausalLM.from_pretrained(LLAMA, device_map="auto", quantization_config=config_quantizacao)

In [29]:
memoria = modelo.get_memory_footprint() / 1e6
print(f"Uso de memória (VRAM): {memoria:,.1f} MB")

Uso de memória (VRAM): 1,012.0 MB


## Inspecionando a Arquitetura do Modelo Transformer

A célula seguinte imprime o objeto `modelo` PyTorch/HuggingFace para o Llama.

Este objeto de modelo é uma Rede Neural implementada com a biblioteca PyTorch baseada na arquitetura Transformer.

Ao observar as camadas da Rede Neural impressas na célula seguinte, note os seguintes pontos principais:

- O modelo é composto por várias camadas encadeadas;
- Existe a camada de **Embedding**: transforma os tokens em vetores multidimensionais (ex: 4.096 dimensões);
- Em seguida, há grupos de **Decoder layers** (camadas decodificadoras). Cada camada contém: (a) Self-Attention (auto-atenção) (b) Multi-Layer Perceptron (MLP) (c) Camadas de normalização;
- No final, há a camada **LM Head**, responsável por gerar os tokens de saída;
- Observe também a indicação de que o modelo foi quantizado em 4 bits.

In [30]:
# Execute esta célula para inspecionar as camadas do modelo

modelo

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear4bit(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm

### Aprofundando o estudo no código do Transformer

Além de inspecionar as camadas do modelo no Python, você pode conferir o código-fonte em PyTorch diretamente no repositório Transformers da Hugging Face:

- Repositório HuggingFace Transformers: https://github.com/huggingface/transformers
- Código do modelo Llama: https://github.com/huggingface/transformers/blob/main/src/transformers/models/llama/modeling_llama.py


In [31]:
# Executando a geração de texto com o modelo

saidas = modelo.generate(entradas, max_new_tokens=80)
saidas[0]

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


tensor([128000, 128006,   9125, 128007,    271,  38766,   1303,  33025,   2696,
            25,   6790,    220,   2366,     18,    198,  15724,   2696,     25,
           220,   1721,   5033,    220,   2366,     21,    271, 128009, 128006,
           882, 128007,    271,    825,     68,  10832,   9115,   2649,    665,
         33050,   3209,   2649,   3429,  10832,  59033,    409,    356,   1188,
         27771,    409,    423,   5670, 128009, 128006,  78191, 128007,    271,
            32,  47391,  40586,  10832,   9115,   2649,   1473,      1,  55218,
          1826,    283,  44676,  28895,  25738,     82,  20847,    309,  93817,
           264,  47615,  19130,    470,  13325,     11,   9427,  12674,    934,
         12329,    336,  93817,    264,  47615,  19130,    470,  13325,     13,
         29124,   8112,    513,  25738,     82,    259,   1924,   7143,  10832,
         59033,    409,    356,   1188,  27771,    409,    423,   5670,     11,
          9427,  25738,     82,  12674, 

In [32]:
# Decodificando os tokens gerados em texto compreensível:

tokenizador.decode(saidas[0])

'<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 01 Aug 2026\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nConte uma piada engraçada para uma sala de Cientistas de Dados<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nAqui vai uma piada:\n\n"Eu estou aqui porque vocês precisam aprender a usar pandas com Python, mas não quiserem aprender a usar pandas com Python. É como se vocês tivessem uma sala de Cientistas de Dados, mas vocês não quisessem que os cientistas de dados entrassem na sala para explicar o'

In [33]:
# Limpeza da memória GPU (VRAM)

del modelo, entradas, tokenizador, saidas
gc.collect()
torch.cuda.empty_cache()

## Notas sobre o envio em tempo real (Streaming)

Utilizamos o utilitário `TextStreamer` do Hugging Face para que as respostas sejam exibidas token por token conforme são geradas.

Substituímos:  
`saidas = modelo.generate(entradas, max_new_tokens=80)`  
Por:  
`streamer = TextStreamer(tokenizador)`  
`saidas = modelo.generate(entradas, max_new_tokens=80, streamer=streamer)`

Também adicionamos o argumento `add_generation_prompt=True` no template do chat para garantir que o modelo gere uma resposta à pergunta, em vez de apenas tentar continuar o prompt do usuário.

In [34]:
# Função utilitária para carregar o modelo e gerar resposta com streaming

def gerar_resposta(modelo_id, mensagens, quantizado=True, max_novos_tokens=80):
    tokenizador = AutoTokenizer.from_pretrained(modelo_id)
    tokenizador.pad_token = tokenizador.eos_token
    ids_entrada = tokenizador.apply_chat_template(mensagens, return_tensors="pt", add_generation_prompt=True).to("cuda")
    mascara_atencao = torch.ones_like(ids_entrada, dtype=torch.long, device="cuda")
    streamer = TextStreamer(tokenizador)
    if quantizado:
        modelo = AutoModelForCausalLM.from_pretrained(modelo_id, quantization_config=config_quantizacao).to("cuda")
    else:
        modelo = AutoModelForCausalLM.from_pretrained(modelo_id).to("cuda")
    saidas = modelo.generate(input_ids=ids_entrada, attention_mask=mascara_atencao, max_new_tokens=max_novos_tokens, streamer=streamer)


In [35]:
gerar_resposta(PHI, mensagens)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

<|user|>Conte uma piada engraçada para uma sala de Cientistas de Dados<|end|><|assistant|>Claro, aqui está uma piada para Cientistas de Dados:

Por que o cientista de dados levou seu laptop ao bar?

Porque ele ouviu que o drink era bem "descritível" e queria garantir que ele pudesse "analisar" o sabor com precisão!<|end|>


## Acesso ao modelo Gemma do Google

O Google também exige que você aceite os termos de uso no HuggingFace antes de usar o Gemma.

Visite a página do modelo para aceitar os termos:
https://huggingface.co/google/gemma-3-270m-it

In [36]:
mensagens = [
    {"role": "user", "content": "Conte uma piada leve e divertida para Cientistas de Dados"}
  ]
gerar_resposta(GEMMA, mensagens, quantizado=False)

Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


<bos><start_of_turn>user
Conte uma piada leve e divertida para Cientistas de Dados<end_of_turn>
<start_of_turn>model
Claro, aqui está uma piada leve e divertida para Cientistas de Dados:

Um cientista de dados está sentada em seu laboratório, com uma folha de papel em sua mão. Ele olha para a folha e diz: "Olá, qual o seu nome?"

A pessoa responde: "Eu sou Ana, eu estou na equipe de pesquisa."

O cientista de


In [37]:
gerar_resposta(QWEN, mensagens)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

<|im_start|>user
Conte uma piada leve e divertida para Cientistas de Dados<|im_end|>
<|im_start|>assistant
Claro! Aqui vai uma piada leve e divertida, feita especialmente para cientistas de dados:

---

**Por que o cientista de dados nunca consegue acertar no alvo no archery?**  
Porque ele só confia em **modelos preditivos** — e o alvo é *muito* ruim em prever o que vai aconte


In [38]:
gerar_resposta(DEEPSEEK, mensagens, quantizado=False, max_novos_tokens=500)

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


<｜begin▁of▁sentence｜><｜User｜>Conte uma piada leve e divertida para Cientistas de Dados<｜Assistant｜><think>
Okay, the user asked for a fun and tasty piada leva with some data science elements. I need to make it both appealing and educational.

First, I'll start with the traditional ingredients: quinoa, potatoes, tomatoes, and spinach. These are all classic and easy to prepare.

Next, I'll add some creative twist by including some data visualization tools like Matplotlib and Seaborn. This shows how data can be used to enhance the dish.

I should highlight the benefits of data science in everyday life, like making decisions faster and understanding trends better. This makes the piada more relatable.

Adding a fun element, maybe some simple Python code to create a visualization. It shows how the user can apply what they've learned.

Finally, I'll wrap it up by mentioning how this piada can be a fun way to learn about data science without needing advanced knowledge.

I think that covers all